# UrbanStyle — nädal 7 grupitöö

## Nädala teema

**Python ja pandas: RFM-kliendisegmenteerimine**

## Nädala eesmärk

Nädala eesmärk on kasutada Pythonit ja pandas't UrbanStyle'i kliendiandmete analüüsimiseks, et:

- laadida ja ühendada vajalikud andmed;
- puhastada analüüsiks kasutatav andmestik;
- arvutada klientidele RFM-näitajad;
- jaotada kliendid segmentidesse;
- visualiseerida tulemused;
- anda Markole andmetel põhinev äritõlgendus ja tegevussoovitused.

## Rollid

| Roll | Rolli pealkiri | Vastutaja |
|---|---|---|
| **Roll A** | Data Loading — andmete laadimine ja ühendamine | **Natalia** |
| **Roll B** | Data Cleaning — andmete puhastamine | **Olga** |
| **Roll C** | RFM Analysis — RFM-arvutused ja kliendisegmendid | **Helen** |
| **Roll D** | Visualization — visualiseerimine ja äritõlgendus | **Kalju** |

Koondfailis on Roll A, Roll B ja Roll C originaaltööd lisatud muutmata kujul. Roll D lisab oma töö lõpus olevasse Roll D struktuuri.


# Roll A — Data Loading

**Vastutaja: Natalia**

Allpool on Roll A parandatud originaaltöö muutmata kujul.


In [1]:
#ROLL A - Data Loading (Andmete laadimine)_NATALIA

import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from supabase import create_client

# 1. Lae muutujad .env failist
load_dotenv(find_dotenv())

url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_KEY")

# 2. Ühendamine ja andmete laadimine Supabase'ist
# 2.1 Funktsioon, mis laadib Supabase'ist kõik read
# Supabase tagastab ühe päringuga maksimaalselt 1000 rida,
# seetõttu laaditakse andmed 1000 rea kaupa.
def get_data(tabel_name):
    data = []
    page_size = 1000
    page = 0

    while True:
        response = (supabase.table(tabel_name).select("*").range(page * page_size,(page + 1) * page_size - 1)
            .execute())
        data.extend(response.data)
        if len(response.data) < page_size:
            break
        page += 1
    return pd.DataFrame(data)

# 2.2. Proovime andmed Supabase'ist laadida
try:
    # Loome ühenduse Supabase'iga
    supabase = create_client(url, key)

    # Laeme kõik müügi- ja kliendiread
    df_sales = get_data("sales")
    df_customers = get_data("customers")

    print("✅ Andmed laeti edukalt Supabase'ist!")
    print("Sales tabeli shape:", df_sales.shape)
    print("Customers tabeli shape:", df_customers.shape)

# 2.3. Kui Supabase ei tööta, proovime CSV-faile
except Exception as e:
    print(f"⚠️ Supabase ei toiminud ({e}), laadime lokaalsetest CSV-failidest...")
    df_sales = pd.read_csv("sales.csv")
    df_customers = pd.read_csv("customers.csv")
    print("✅ Andmed laeti CSV-failidest!\n")
    print("Sales tabeli shape:", df_sales.shape)
    print("Customers tabeli shape:", df_customers.shape)

✅ Andmed laeti edukalt Supabase'ist!
Sales tabeli shape: (10118, 12)
Customers tabeli shape: (3150, 9)


In [2]:
# 3. Sales tabeli kontroll
print("--- 1. SALES TABEL ---")
print("Sales tabeli shape:", df_sales.shape)
print(df_sales.head())

--- 1. SALES TABEL ---
Sales tabeli shape: (10118, 12)
   id  sale_id        invoice_id            sale_date  customer_id  \
0   1        1  INV-202301-00001  2023-01-10T00:00:00       2588.0   
1   2        2  INV-202301-00002  2023-01-16T00:00:00       4338.0   
2   3        3  INV-202301-00003  2023-01-05T00:00:00       4673.0   
3   4        4  INV-202301-00004  2023-01-02T00:00:00       4677.0   
4   5        5  INV-202301-00005  2023-01-13T00:00:00       2390.0   

   product_id  quantity  unit_price  total_price channel store_location  \
0        1274         2      234.79       469.58    pood        Tallinn   
1        1207         2      241.13       482.26    pood          Pärnu   
2        1264         1      258.46       221.19    pood          Pärnu   
3        1341         3       45.21       135.63    pood          Tartu   
4        1284         1       99.57        99.57    pood          Tartu   

  payment_method  
0          kaart  
1      järelmaks  
2      järelmaks

In [3]:
# 4. Customers tabeli kontroll
print("\n--- 2. CUSTOMERS TABEL ---")
print("Customers tabeli shape:", df_customers.shape)
print(df_customers.head())


--- 2. CUSTOMERS TABEL ---
Customers tabeli shape: (3150, 9)
   customer_id first_name last_name                   email           phone  \
0         2001        Eha       Aas        eha.aas@telia.ee  +372 8713 1455   
1         2002      Aivar      Kõiv  aivar.koiv@outlook.com  +372 8943 8684   
2         2003      Maris    Rebane   maris.rebane@telia.ee  +372 5918 5726   
3         2005      Raivo    Koppel  raivo.koppel@yahoo.com  +372 5298 4365   
4         2006       Aili      Must                     NaN  +372 5444 0491   

       city registration_date loyalty_tier  birth_year  
0   Tallinn        2024-02-27          NaN        1973  
1  Haapsalu        2025-01-09       bronze        1988  
2     Tartu        2021-02-03          NaN        1999  
3   Tallinn        2023-05-22       bronze        2004  
4     Pärnu        2022-01-04          NaN        1986  


In [4]:
# 5. Liidame tabelid
df = pd.merge(df_sales, df_customers, on="customer_id", how="left")

In [5]:
print("\n--- 3. LIIDATUD DATAFRAME (df) ---")
print("Liidatud tabeli shape:", df.shape)
print("\nVeergude tüübid (dtypes):")
print(df.dtypes)
print("\nEsimesed 5 rida (head):")
print(df.head())


--- 3. LIIDATUD DATAFRAME (df) ---
Liidatud tabeli shape: (10118, 20)

Veergude tüübid (dtypes):
id                     int64
sale_id                int64
invoice_id               str
sale_date                str
customer_id          float64
product_id             int64
quantity               int64
unit_price           float64
total_price          float64
channel                  str
store_location           str
payment_method           str
first_name               str
last_name                str
email                    str
phone                    str
city                     str
registration_date        str
loyalty_tier             str
birth_year           float64
dtype: object

Esimesed 5 rida (head):
   id  sale_id        invoice_id            sale_date  customer_id  \
0   1        1  INV-202301-00001  2023-01-10T00:00:00       2588.0   
1   2        2  INV-202301-00002  2023-01-16T00:00:00       4338.0   
2   3        3  INV-202301-00003  2023-01-05T00:00:00       4673.0   
3  

In [6]:
# Seadistame Pandas'i kuvama kõiki veerge
pd.set_option('display.max_columns', None)

print("--- LIIDATUD TABELI ESIMESED READ ---")
display(df.head())

print("\n--- ANDMETE ÜLDULEVAATUS ---")
df.info()

--- LIIDATUD TABELI ESIMESED READ ---


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0



--- ANDMETE ÜLDULEVAATUS ---
<class 'pandas.DataFrame'>
RangeIndex: 10118 entries, 0 to 10117
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 10118 non-null  int64  
 1   sale_id            10118 non-null  int64  
 2   invoice_id         10118 non-null  str    
 3   sale_date          10118 non-null  str    
 4   customer_id        9130 non-null   float64
 5   product_id         10118 non-null  int64  
 6   quantity           10118 non-null  int64  
 7   unit_price         10118 non-null  float64
 8   total_price        10118 non-null  float64
 9   channel            10118 non-null  str    
 10  store_location     6656 non-null   str    
 11  payment_method     10118 non-null  str    
 12  first_name         9130 non-null   str    
 13  last_name          9130 non-null   str    
 14  email              8174 non-null   str    
 15  phone              9130 non-null   str    
 16  cit

# Roll B — Data Cleaning

**Vastutaja: Olga**


In [7]:
print(df.shape)

(10118, 20)


In [8]:
print("Esialgne shape:", df.shape)

# Duplikaadid
print("Duplikaadid:", df.duplicated(subset=['invoice_id']).sum())
df = df.drop_duplicates(subset=['invoice_id'], keep='first')

# NULL väärtused
print("\nNULL väärtused:")
print(df.isnull().sum())

df = df.dropna(subset=['customer_id', 'sale_date', 'total_price'])

# Kuupäev datetime-ks
df['sale_date'] = pd.to_datetime(df['sale_date'])

# Negatiivsed hinnad
df = df[df['total_price'] > 0]

# Puhastusraport
print("\n===== PUHASTUSRAPORT =====")
print("Lõplik shape:", df.shape)
print("Unikaalseid kliente:", df['customer_id'].nunique())
print("Kuupäevavahemik:",
      df['sale_date'].min(),
      "kuni",
      df['sale_date'].max())

df.head()

Esialgne shape: (10118, 20)
Duplikaadid: 0

NULL väärtused:
id                      0
sale_id                 0
invoice_id              0
sale_date               0
customer_id           988
product_id              0
quantity                0
unit_price              0
total_price             0
channel                 0
store_location       3462
payment_method          0
first_name            988
last_name             988
email                1944
phone                 988
city                  988
registration_date     988
loyalty_tier         4660
birth_year            988
dtype: int64

===== PUHASTUSRAPORT =====
Lõplik shape: (8950, 20)
Unikaalseid kliente: 2540
Kuupäevavahemik: 2023-01-01 00:00:00 kuni 2026-06-28 00:00:00


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,2,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,3,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,4,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,5,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0


In [9]:
print("\n--- 3. LIIDATUD DATAFRAME (df) ---")
print("Liidatud tabeli shape:", df.shape)
print("\nVeergude tüübid (dtypes):")
print(df.dtypes)
print("\nEsimesed 5 rida (head):")
print(df.head())


--- 3. LIIDATUD DATAFRAME (df) ---
Liidatud tabeli shape: (8950, 20)

Veergude tüübid (dtypes):
id                            int64
sale_id                       int64
invoice_id                      str
sale_date            datetime64[us]
customer_id                 float64
product_id                    int64
quantity                      int64
unit_price                  float64
total_price                 float64
channel                         str
store_location                  str
payment_method                  str
first_name                      str
last_name                       str
email                           str
phone                           str
city                            str
registration_date               str
loyalty_tier                    str
birth_year                  float64
dtype: object

Esimesed 5 rida (head):
   id  sale_id        invoice_id  sale_date  customer_id  product_id  \
0   1        1  INV-202301-00001 2023-01-10       2588.0        1274   


In [10]:
# Seadistame Pandas'i kuvama kõiki veerge
pd.set_option('display.max_columns', None)

print("--- LIIDATUD TABELI ESIMESED READ ---")
display(df.head())

print("\n--- ANDMETE ÜLDULEVAATUS ---")
df.info()

--- LIIDATUD TABELI ESIMESED READ ---


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,2,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,3,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,4,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,5,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0



--- ANDMETE ÜLDULEVAATUS ---
<class 'pandas.DataFrame'>
Index: 8950 entries, 0 to 10116
Data columns (total 20 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id                 8950 non-null   int64         
 1   sale_id            8950 non-null   int64         
 2   invoice_id         8950 non-null   str           
 3   sale_date          8950 non-null   datetime64[us]
 4   customer_id        8950 non-null   float64       
 5   product_id         8950 non-null   int64         
 6   quantity           8950 non-null   int64         
 7   unit_price         8950 non-null   float64       
 8   total_price        8950 non-null   float64       
 9   channel            8950 non-null   str           
 10  store_location     5896 non-null   str           
 11  payment_method     8950 non-null   str           
 12  first_name         8950 non-null   str           
 13  last_name          8950 non-null   str          

In [11]:
1 + 1

2

# Nädal 7 — Roll C: RFM kliendisegmenteerimine

**Vastutaja: Helen**

## Roll ja töö ulatus

**Roll C — RFM Analysis**

- Recency, Frequency ja Monetary arvutamine;
- R-, F- ja M-skooride määramine;
- RFM-koondskoori loomine;
- baastaseme kliendisegmentide määramine;
- segmentide kokkuvõte;
- edasijõudnute taseme kaalutud skoor, detailsemad segmendid ja CSV eksport.

### Sisend

Roll B annab puhastatud pandas DataFrame'i nimega `df`.

## 1. Teegi import ja Roll B sisend

Pandas imporditakse RFM-arvutuste tegemiseks.

> Enne järgmiste lahtrite käivitamist peab Roll A andmed imporditud ning roll B puhastatud DataFrame `df` olema notebook'is olemas.

In [12]:
import pandas as pd

# Roll B väljund:
# df = puhastatud müügi- ja kliendiandmete DataFrame

df.head()

,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,1,1,INV-202301-00001,2023-01-10,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,Paju,NaN,+372 5429 0294,Tallinn,2022-07-28,bronze,1997.0
1,2,2,INV-202301-00002,2023-01-16,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,Luik,merle.luik@mail.ee,+372 5150 1812,Tallinn,2020-09-22,NaN,1996.0
2,3,3,INV-202301-00003,2023-01-05,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,Saar,liina.saar@gmail.com,+372 8809 7990,Tallinn,2020-03-31,silver,1973.0
3,4,4,INV-202301-00004,2023-01-02,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,Pihl,aili.pihl@yahoo.com,+372 8375 4888,Narva,2021-10-08,gold,1972.0
4,5,5,INV-202301-00005,2023-01-13,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,Lill,triin.lill@telia.ee,+372 5378 0596,Tartu,2021-04-09,NaN,1996.0


## 2. Baastase — RFM-mõõdikute arvutamine

### 2.1. Määra viitekuupäev: 

In [13]:
today = pd.to_datetime("2025-02-28")

print("RFM viitekuupäev:", today)

RFM viitekuupäev: 2025-02-28 00:00:00


### 2.2. Recency

Recency näitab päevade arvu kliendi viimasest ostust viitekuupäevani.

Madalam Recency väärtus on parem.

In [14]:
recency = (
    df.groupby("customer_id")["sale_date"]
    .max()
    .reset_index()
)

recency.columns = [
    "customer_id",
    "last_purchase_date"
]

recency["recency_days"] = (
    today - recency["last_purchase_date"]
).dt.days

recency.head()

,customer_id,last_purchase_date,recency_days
0,2001.0,2024-11-29,91
1,2004.0,2024-12-19,71
2,2005.0,2024-10-03,148
3,2006.0,2023-11-09,477
4,2007.0,2025-01-30,29


### 2.3. Frequency

Frequency näitab kliendi ostude arvu.

In [15]:
frequency = (
    df.groupby("customer_id")["sale_id"]
    .count()
    .reset_index()
)

frequency.columns = [
    "customer_id",
    "frequency"
]

frequency.head()

,customer_id,frequency
0,2001.0,2
1,2004.0,2
2,2005.0,4
3,2006.0,1
4,2007.0,1


### 2.4. Monetary

Monetary näitab kliendi kogukulutust.

In [16]:
monetary = (
    df.groupby("customer_id")["total_price"]
    .sum()
    .reset_index()
)

monetary.columns = [
    "customer_id",
    "monetary_value"
]

monetary.head()

,customer_id,monetary_value
0,2001.0,203.92
1,2004.0,1198.56
2,2005.0,959.60
3,2006.0,327.06
4,2007.0,318.63


### 2.5. RFM-tabeli ühendamine

Recency, Frequency ja Monetary ühendatakse üheks kliendipõhiseks tabeliks.

In [17]:
rfm = (
    recency[["customer_id", "recency_days"]]
    .merge(
        frequency,
        on="customer_id"
    )
    .merge(
        monetary,
        on="customer_id"
    )
)

rfm.head()

,customer_id,recency_days,frequency,monetary_value
0,2001.0,91,2,203.92
1,2004.0,71,2,1198.56
2,2005.0,148,4,959.60
3,2006.0,477,1,327.06
4,2007.0,29,1,318.63


## 3. Baastase — RFM-skoorid

Iga RFM-mõõdik hinnatakse kvintiilide alusel skaalal 1–5.

- Recency: madalam väärtus saab kõrgema skoori.
- Frequency: kõrgem väärtus saab kõrgema skoori.
- Monetary: kõrgem väärtus saab kõrgema skoori.

In [18]:
rfm["R_score"] = pd.qcut(
    rfm["recency_days"],
    5,
    labels=[5, 4, 3, 2, 1]
)

rfm["F_score"] = pd.qcut(
    rfm["frequency"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm["M_score"] = pd.qcut(
    rfm["monetary_value"],
    5,
    labels=[1, 2, 3, 4, 5]
)

rfm["R_score"] = rfm["R_score"].astype(int)
rfm["F_score"] = rfm["F_score"].astype(int)
rfm["M_score"] = rfm["M_score"].astype(int)

rfm["RFM_Score"] = (
    rfm["R_score"]
    + rfm["F_score"]
    + rfm["M_score"]
)

rfm.head()

,customer_id,recency_days,frequency,monetary_value,R_score,F_score,M_score,RFM_Score
0,2001.0,91,2,203.92,4,1,1,6
1,2004.0,71,2,1198.56,4,1,4,9
2,2005.0,148,4,959.60,3,4,3,10
3,2006.0,477,1,327.06,1,1,1,3
4,2007.0,29,1,318.63,5,1,1,7


## 4. Baastase — viis kliendisegmenti

Juhendi baastaseme segmendid:

| RFM-skoor | Segment |
|---:|---|
| 13–15 | VIP Champions |
| 10–12 | Loyal |
| 7–9 | Potential |
| 4–6 | At Risk |
| 3 | Lost |

In [19]:
def segment_customer(row):
    if row["RFM_Score"] >= 13:
        return "VIP Champions"
    elif row["RFM_Score"] >= 10:
        return "Loyal"
    elif row["RFM_Score"] >= 7:
        return "Potential"
    elif row["RFM_Score"] >= 4:
        return "At Risk"
    else:
        return "Lost"


rfm["Segment"] = rfm.apply(
    segment_customer,
    axis=1
)

rfm.head()

,customer_id,recency_days,frequency,monetary_value,R_score,F_score,M_score,RFM_Score,Segment
0,2001.0,91,2,203.92,4,1,1,6,At Risk
1,2004.0,71,2,1198.56,4,1,4,9,Potential
2,2005.0,148,4,959.60,3,4,3,10,Loyal
3,2006.0,477,1,327.06,1,1,1,3,Lost
4,2007.0,29,1,318.63,5,1,1,7,Potential


## 5. Baastaseme kokkuvõte

Kokkuvõte näitab iga segmendi klientide arvu ja osakaalu.

In [20]:
segment_summary = (
    rfm["Segment"]
    .value_counts()
    .rename_axis("Segment")
    .reset_index(name="customers")
)

segment_summary["customer_share_pct"] = (
    segment_summary["customers"]
    / len(rfm)
    * 100
)

segment_summary["customer_share_pct"] = (
    segment_summary["customer_share_pct"]
    .round(2)
)

segment_summary

,Segment,customers,customer_share_pct
0,Potential,759,29.88
1,Loyal,679,26.73
2,At Risk,529,20.83
3,VIP Champions,455,17.91
4,Lost,118,4.65


### Baastaseme kvaliteedikontroll

Kontrollitakse juhendis nõutud tingimusi:

- RFM-väärtused on arvutatud;
- skoorid jäävad vahemikku 1–5;
- iga klient sai segmendi;
- kokkuvõte näitab klientide arvu ja osakaalu.

In [21]:
print("Kliente RFM-tabelis:", len(rfm))
print("Segmendita kliente:", rfm["Segment"].isna().sum())

print("\nSkooride vahemikud:")
print(
    rfm[
        ["R_score", "F_score", "M_score"]
    ].agg(["min", "max"])
)

print(
    "\nKlientide osakaal kokku:",
    round(segment_summary["customer_share_pct"].sum(), 2),
    "%"
)

Kliente RFM-tabelis: 2540
Segmendita kliente: 0

Skooride vahemikud:
     R_score  F_score  M_score
min        1        1        1
max        5        5        5

Klientide osakaal kokku: 100.0 %


# Edasijõudnute tase

Juhendi vabatahtlik edasijõudnute osa sisaldab:

1. Monetary kahekordse kaaluga skoori;
2. kuut detailsemat kliendisegmenti;
3. segmentide eksporti CSV-failina.

## 6. Kaalutud RFM-skoor

Monetary saab kahekordse kaalu, sest kliendi kulutus on Marko jaoks olulisem.

Kaalutud skoor:

`R_score + F_score + 2 × M_score`

In [22]:
rfm["Weighted_RFM_Score"] = (
    rfm["R_score"]
    + rfm["F_score"]
    + 2 * rfm["M_score"]
)

rfm[
    [
        "customer_id",
        "R_score",
        "F_score",
        "M_score",
        "RFM_Score",
        "Weighted_RFM_Score"
    ]
].head()

,customer_id,R_score,F_score,M_score,RFM_Score,Weighted_RFM_Score
0,2001.0,4,1,1,6,7
1,2004.0,4,1,4,9,13
2,2005.0,3,4,3,10,13
3,2006.0,1,1,1,3,4
4,2007.0,5,1,1,7,8


## 7. Detailsemad segmendid

Juhendi edasijõudnute segmendid:

| RFM-skoor | Segment | Tegevus |
|---:|---|---|
| 13–15 | VIP Champions | Early access, VIP sooduskoodid |
| 11–12 | Loyal Customers | Lojaalsusprogramm, preemiad |
| 9–10 | Regular Customers | Cross-sell kampaaniad |
| 7–8 | New Customers | Onboarding, welcome-sari |
| 5–6 | At Risk | Win-back kampaania, personaliseeritud e-mail |
| 3–4 | Lost | Viimane katse, suur soodustus |

In [23]:
def assign_advanced_segment(row):
    if row["RFM_Score"] >= 13:
        return "VIP Champions"
    elif row["RFM_Score"] >= 11:
        return "Loyal Customers"
    elif row["RFM_Score"] >= 9:
        return "Regular Customers"
    elif row["RFM_Score"] >= 7:
        return "New Customers"
    elif row["RFM_Score"] >= 5:
        return "At Risk"
    else:
        return "Lost"


rfm["Advanced_Segment"] = rfm.apply(
    assign_advanced_segment,
    axis=1
)

rfm.head()

,customer_id,recency_days,frequency,monetary_value,R_score,F_score,M_score,RFM_Score,Segment,Weighted_RFM_Score,Advanced_Segment
0,2001.0,91,2,203.92,4,1,1,6,At Risk,7,At Risk
1,2004.0,71,2,1198.56,4,1,4,9,Potential,13,Regular Customers
2,2005.0,148,4,959.60,3,4,3,10,Loyal,13,Regular Customers
3,2006.0,477,1,327.06,1,1,1,3,Lost,4,Lost
4,2007.0,29,1,318.63,5,1,1,7,Potential,8,New Customers


## 8. Edasijõudnute segmentide kokkuvõte

In [24]:
advanced_segment_summary = (
    rfm["Advanced_Segment"]
    .value_counts()
    .rename_axis("Advanced_Segment")
    .reset_index(name="customers")
)

advanced_segment_summary["customer_share_pct"] = (
    advanced_segment_summary["customers"]
    / len(rfm)
    * 100
)

advanced_segment_summary["customer_share_pct"] = (
    advanced_segment_summary["customer_share_pct"]
    .round(2)
)

advanced_segment_summary

,Advanced_Segment,customers,customer_share_pct
0,Regular Customers,512,20.16
1,New Customers,511,20.12
2,VIP Champions,455,17.91
3,Loyal Customers,415,16.34
4,At Risk,391,15.39
5,Lost,256,10.08


## 9. Tulemuste eksport

Juhendi järgi salvestatakse segmendid faili `rfm_segments.csv`, et Marko saaks tulemuse turundusmeeskonnale edastada.

In [25]:
rfm.to_csv(
    "rfm_segments.csv",
    index=False
)

print("Fail salvestatud: rfm_segments.csv")

Fail salvestatud: rfm_segments.csv


## 10. Roll C väljund

Roll C annab Roll D-le edasi DataFrame'i `rfm`, mis sisaldab:

- `customer_id`;
- `recency_days`;
- `frequency`;
- `monetary_value`;
- R-, F- ja M-skoore;
- baastaseme RFM-koondskoori ja segmenti;
- edasijõudnute kaalutud skoori ja detailsemat segmenti.

Roll D kasutab seda tabelit visualiseerimiseks ja äritõlgenduse koostamiseks.

# Roll D — Visualization

**Vastutaja: Kalju**

Roll D lisab siia visualiseerimise, tulemuste tõlgendamise ja tegevussoovitused.


## Roll D tööstruktuur

### 1. Segmentide jaotus

Lisa visualiseering, mis näitab klientide arvu RFM-segmentide lõikes.

### 2. Recency ja Monetary võrdlus

Lisa hajuvusdiagramm, mis näitab klientide ostuaktiivsuse ja rahalise väärtuse seost.

### 3. TOP 10 VIP-klienti

Lisa visualiseering kümnest suurima väärtusega VIP-kliendist.

### 4. Äritõlgendus Markole

Kirjelda lühidalt:

- mitu VIP-klienti tuvastati;
- kui suur on VIP-klientide käibeosakaal;
- mitu klienti kuulub riskisegmenti;
- milline segment vajab esmast tähelepanu;
- mida tulemused tähendavad kliendisuhete juhtimise jaoks.

### 5. Tegevussoovitused

Lisa 2–3 konkreetset andmetel põhinevat soovitust.
